In [16]:
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
# from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

#Base Models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

# Ensemble Models
from sklearn.ensemble import (
    VotingClassifier, StackingClassifier, BaggingClassifier,
    GradientBoostingClassifier, AdaBoostClassifier
)

# 2. DATA LOADING & SPLITTING
df=load_breast_cancer()
X = df.data
y=df.target

In [17]:
print(df.feature_names)

['mean radius' 'mean texture' 'mean perimeter' 'mean area'
 'mean smoothness' 'mean compactness' 'mean concavity'
 'mean concave points' 'mean symmetry' 'mean fractal dimension'
 'radius error' 'texture error' 'perimeter error' 'area error'
 'smoothness error' 'compactness error' 'concavity error'
 'concave points error' 'symmetry error' 'fractal dimension error'
 'worst radius' 'worst texture' 'worst perimeter' 'worst area'
 'worst smoothness' 'worst compactness' 'worst concavity'
 'worst concave points' 'worst symmetry' 'worst fractal dimension']


In [18]:
print(df.target_names)

['malignant' 'benign']


In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=True, random_state=42
)

In [20]:
# 3. PREPROCESSING (Scaling and Dimensionality Reduction)
scale = MinMaxScaler()
X_train_scaled = scale.fit_transform(X_train)
X_test_scaled = scale.transform(X_test)

In [21]:
pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

In [22]:
# 4. MODEL INITIALIZATIOn
lor=LogisticRegression()

# CalibratedClassifierCV replaces SVC(probability=True) to fix the warning
svm=SVC(probability=True)
#svm = CalibratedClassifierCV(SVC(), ensemble=False)
knn=KNeighborsClassifier(n_neighbors=5)
tree = DecisionTreeClassifier(max_depth=3)

In [23]:
## 5. MODEL DICTIONARY (DRY Principle: Don't Repeat Yourself)
models = {
    "Logistic Regression": lor,
    "SVM": svm,
    "KNN": knn,
    "Soft Voting": VotingClassifier(estimators=[("LR", lor), ("SVM", svm), ("KNN", knn)], voting="soft"),
    "Stacking": StackingClassifier(estimators=[("LR", lor), ("SVM", svm), ("KNN", knn)], final_estimator=tree),
    "Bagging (SVM)": BaggingClassifier(estimator=svm, n_estimators=10),
    "Gradient Boosting": GradientBoostingClassifier(learning_rate=0.1, n_estimators=10, random_state=42),
    "AdaBoost": AdaBoostClassifier(n_estimators=10, learning_rate=0.1, random_state=42)
}

In [24]:
# 6. TRAINING AND EVALUATION LOOP
for name, model in models.items():
    model.fit(X_train_pca, y_train)
    y_pred = model.predict(X_test_pca)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"{name:<20}: {accuracy:.4f}")

Logistic Regression : 0.9649
SVM                 : 0.9561
KNN                 : 0.9649
Soft Voting         : 0.9561
Stacking            : 0.9561
Bagging (SVM)       : 0.9561
Gradient Boosting   : 0.9474
AdaBoost            : 0.9211


Voting

In [25]:
# from sklearn.ensemble import VotingClassifier
# votecls=VotingClassifier(estimators=[("LogisticRegression",lor),("SVM",svm),("KNN",knn)],voting="hard")
# votecls.fit(x_train_pca,y_train)
# y_pred=votecls.predict(x_test_pca)
# print(accuracy_score(y_test,y_pred))

In [26]:
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score

votecls=VotingClassifier(estimators=[("LogisticRegression",lor),("SVM",svm),("KNN",knn)],voting="soft")
votecls.fit(X_train_pca,y_train)
y_pred=votecls.predict(X_test_pca)
print(accuracy_score(y_test,y_pred))

0.956140350877193


Stacking

In [27]:
from sklearn.ensemble import StackingClassifier
from sklearn.tree import DecisionTreeClassifier

stack=StackingClassifier(estimators=[("LogisticRegression",lor),("SVM",svm),("KNN",knn)],final_estimator=DecisionTreeClassifier(max_depth=3))
stack.fit(X_train_pca,y_train)
y_pred=stack.predict(X_test_pca)
print(accuracy_score(y_test,y_pred))

0.956140350877193


Bagging

In [28]:
from sklearn.ensemble import BaggingClassifier

bagg=BaggingClassifier(estimator=svm,n_estimators=10,random_state=42)
bagg.fit(X_train_pca,y_train)
y_pred=bagg.predict(X_test_pca)
print(accuracy_score(y_test,y_pred))

0.956140350877193


Gradient Boosting

In [29]:
from sklearn.ensemble import GradientBoostingClassifier

gboost=GradientBoostingClassifier(learning_rate=0.1,n_estimators=10,random_state=42)
gboost.fit(X_train_pca,y_train)
y_pred=gboost.predict(X_test_pca)
print(accuracy_score(y_test,y_pred))

0.9473684210526315


AdaBoosting

In [30]:
from sklearn.ensemble import AdaBoostClassifier

aboost=AdaBoostClassifier(n_estimators=10,learning_rate=0.1,random_state=42)
aboost.fit(X_train_pca,y_train)
y_pred=aboost.predict(X_test_pca)
print(accuracy_score(y_test,y_pred))

0.9210526315789473


Extreme Gradient Boosting

In [ ]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=100, 
    learning_rate=0.1, 
    eval_metric='logloss', 
    random_state=42
)
xgb_model.fit(X_train_pca, y_train)
y_pred = xgb_model.predict(X_test_pca)
accuracy = accuracy_score(y_test, y_pred)

print(f"XGBoost Accuracy: {accuracy:.4f}")

XGBoost Accuracy: 0.9737
